# Sync Work Field Curations

Syncs curations of a work's own fields (oxjob #2242; first `is_retracted`) from the users Heroku Postgres database to a local Databricks Delta table, and rebuilds the view CreateWorksBase reads. Runs in `walden_end2end.yaml` before `Works_Base`.

Jason, 2026-11-14: every correction goes through the one central corrections table (users-api `curations`), never a side table. This replaces `openalex.works.retraction_flag_overrides` (oxjob #1500).

**Source**: `openalex_users.public.work_field_curations`, a typed view over `openalex_users.public.curations` (users-api migration 124; 127 adds removes): one row per curation, `entity='works'`, `action` `replace` or `remove`, `property` one of the curatable fields, `work_id` the bare work id.

**Target**: `openalex.works.work_field_curations` (the log), and the view `openalex.works.work_curated_fields`: one row per curated work, the latest curation per (work, field) by `created DESC, curation_id DESC`, one typed column per field, NULL where no curation set it. A remove (oxjob #1103: `abstract`, `biblio.*`) leaves its column NULL and names the field in `curated_fields`, so CreateWorksBase serves NULL (as ApplyLocationCurations does for location removes).

**Fields:** `is_retracted` (#2242), `type` (#1056), `is_xpac` (#1125), `abstract` and `biblio.*` (#1103), `publication_date` (#1132), `title` (#1162), `language` (#1703), `is_pulled` (#1186); author names (#1156) have their own view. **Adding a field** (as #1703 did for `language`): users-api `work_field_curation.WORK_FIELDS` (the export view selects by shape, so no migration); here, one more `MAX(CASE ...)` column in the view; in CreateWorksBase, `COALESCE(wcf.wcf_<field>, <computed>)`. Steps: oxjob #2242 PLAN.md.

Rows leave Postgres only when an account is deleted (oxjob #1436); they leave this table through `NOT MATCHED BY SOURCE`, and the decline guardrail allows for rows owned by deleted accounts and refuses the clause on any other suspicious empty or short source.


## Create target table (idempotent)

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS openalex.works.work_field_curations (
    curation_id        STRING NOT NULL,
    work_id            BIGINT NOT NULL,
    property           STRING NOT NULL,
    action             STRING NOT NULL,
    value              STRING,
    user_id            STRING,
    created            TIMESTAMP,
    updated_datetime   TIMESTAMP
)
USING DELTA
CLUSTER BY (work_id)


## Sync work field curations

In [ ]:
%sql
-- Empty-when-target-nonempty fails unconditionally; decline check is overridable.
-- (Both 0 on startup is legitimate and doesn't fail.)
DECLARE OR REPLACE VARIABLE new_count BIGINT;
DECLARE OR REPLACE VARIABLE current_count BIGINT;
DECLARE OR REPLACE VARIABLE allowed_decline BIGINT DEFAULT 10;

-- oxjob #1436: deleting an account deletes its curations in users-api, and
-- users-api records the account in deleted_users. Rows here owned by such
-- accounts are expected to go, so they widen the allowed decline; any other
-- drop still trips the guardrail.
SET VARIABLE allowed_decline = 10 + (
  SELECT COUNT(*) FROM openalex.works.work_field_curations t
  WHERE t.user_id IN (SELECT user_id FROM openalex_users.public.deleted_users)
);

SET VARIABLE new_count = (SELECT COUNT(*) FROM openalex_users.public.work_field_curations);
SET VARIABLE current_count = (SELECT COUNT(*) FROM openalex.works.work_field_curations);

SELECT
  new_count AS new_curations,
  current_count AS current_curations,
  openalex.works.e2e_cap('guardrails_override', 'work_field_curations').cap AS preclear_cap;  -- oxjob #2005: a planned decline is named on the pre-clear row for the night (max_loss work_field_curations)

SELECT CASE
  WHEN current_count > 0 AND new_count = 0
  THEN RAISE_ERROR(CONCAT(
    'GUARDRAIL FAILED: source has 0 work field curations but the target holds ',
    CAST(current_count AS STRING),
    ' rows. Aborting to prevent mass delete. Investigate the source view before re-running.'))
END;

SELECT CASE
  WHEN current_count > 0
   AND new_count < current_count - allowed_decline
   AND NOT openalex.works.e2e_gate('guardrails_override', 'work_field_curations', current_count - new_count, allowed_decline,
                                   :guardrails_override)  -- within the allowed decline or the row's cap (oxjob #2005)
  THEN RAISE_ERROR(CONCAT(
    'GUARDRAIL FAILED: source work field curation count declined by ',
    CAST(current_count - new_count AS STRING),
    ' (', CAST(new_count AS STRING), ' vs ', CAST(current_count AS STRING),
    '), exceeding allowed decline of ', CAST(allowed_decline AS STRING),
    '. A planned decline: name it on the night''s pre-clear row (scripts/preclear_e2e.py --raise --max-loss work_field_curations=N), then repair; a manual run-now: guardrails_override=true.'))
END;


In [ ]:
%sql
MERGE INTO openalex.works.work_field_curations AS target
USING (
    SELECT
        curation_id, work_id, property, action, value, user_id, created,
        CURRENT_TIMESTAMP() AS updated_datetime
    FROM openalex_users.public.work_field_curations
) AS source
ON target.curation_id = source.curation_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
WHEN NOT MATCHED BY SOURCE THEN DELETE


## The view CreateWorksBase reads

One row per curated work: the latest curation per (work, field) wins, one typed column per field (NULL = no curation for that field), and `curated_fields` naming the fields set. A plain view: a handful to a few thousand rows.


In [ ]:
%sql
CREATE OR REPLACE VIEW openalex.works.work_curated_fields
COMMENT 'The latest work field curation per (work, field), one row per curated work, one typed column per field (SyncWorkFieldCurations, oxjob #2242). CreateWorksBase lays these over the computed values.'
AS
WITH latest AS (
  SELECT work_id, property, action, value
  FROM openalex.works.work_field_curations
  WHERE action IN ('replace', 'remove') -- oxjob #1103: a remove clears abstract or a biblio field
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY work_id, property
    ORDER BY created DESC, curation_id DESC
  ) = 1
)
SELECT
  work_id,
  -- replace only for these four: users-api takes no remove for them, and a stray one must not read as false / 'null'
  MAX(CASE WHEN property = 'is_retracted' AND action = 'replace' THEN value = 'true' END) AS is_retracted,
  MAX(CASE WHEN property = 'type' AND action = 'replace' THEN value END)                  AS type, -- oxjob #1056
  MAX(CASE WHEN property = 'is_xpac' AND action = 'replace' THEN value = 'true' END)      AS is_xpac, -- oxjob #1125
  MAX(CASE WHEN property = 'is_pulled' AND action = 'replace' THEN value = 'true' END)    AS is_pulled, -- oxjob #1186 (replace only; wins over every pulled signal in CreateWorksBase)
  -- oxjob #1103: a replace's value; NULL for a remove (the field is in curated_fields either way)
  MAX(CASE WHEN property = 'biblio.volume' AND action = 'replace' THEN value END)     AS biblio_volume,
  MAX(CASE WHEN property = 'biblio.issue' AND action = 'replace' THEN value END)      AS biblio_issue,
  MAX(CASE WHEN property = 'biblio.first_page' AND action = 'replace' THEN value END) AS biblio_first_page,
  MAX(CASE WHEN property = 'biblio.last_page' AND action = 'replace' THEN value END)  AS biblio_last_page,
  MAX(CASE WHEN property = 'publication_date' AND action = 'replace' THEN value END) AS publication_date, -- oxjob #1132
  MAX(CASE WHEN property = 'title' AND action = 'replace' THEN value END)           AS title, -- oxjob #1162 (replace only)
  MAX(CASE WHEN property = 'language' AND action = 'replace' THEN value END)        AS language, -- oxjob #1703 (replace only)
  MAX(CASE WHEN property = 'byline' AND action = 'replace' THEN value END)          AS byline, -- oxjob #1815: JSON list of {name, orcid}, as printed (replace only)
  collect_set(property)                                            AS curated_fields
FROM latest
GROUP BY work_id


In [ ]:
%sql
-- oxjob #1156: author-name curations, keyed by the name the work serves (property raw_author_name["<served name>"], value
-- the corrected name). One row per (work, served name), the latest curation winning by created DESC, curation_id DESC,
-- as for every field. CreateWorksEnriched lays these over the served authorships. A plain view: a handful of rows.
-- oxjob #1444: action 'remove' on the same slot drops the seat (to_name NULL); replace and remove compete for latest.
CREATE OR REPLACE VIEW openalex.works.work_curated_author_names
COMMENT 'The latest author-name curation per (work, served name) (SyncWorkFieldCurations, oxjob #1156, #1444). CreateWorksEnriched replaces from_name with to_name in served authorships, or drops the seat when action is remove.'
AS
SELECT
  work_id,
  regexp_extract(property, '^raw_author_name\\["(.+)"\\]$', 1) AS from_name,
  CASE WHEN action = 'replace' THEN value END                   AS to_name,
  action
FROM openalex.works.work_field_curations
WHERE left(property, 17) = 'raw_author_name["'
  AND (action = 'remove' OR (action = 'replace' AND value IS NOT NULL))
QUALIFY ROW_NUMBER() OVER (PARTITION BY work_id, property ORDER BY created DESC, curation_id DESC) = 1

## Verify

In [ ]:
%sql
SELECT
  (SELECT COUNT(*) FROM openalex.works.work_field_curations) AS work_field_curation_rows,
  COUNT(*)                                                    AS curated_works,
  COUNT_IF(is_retracted)                                      AS curated_retracted,
  COUNT_IF(NOT is_retracted)                                  AS curated_not_retracted,
  (SELECT MAX(updated_datetime) FROM openalex.works.work_field_curations) AS last_sync
FROM openalex.works.work_curated_fields


In [ ]:
%sql
-- oxjob #1569: a seat's affiliation strings, keyed by the name the work serves (users-api property
-- raw_affiliation_strings["<served name>"], value a JSON array of strings; [] clears the seat). One row per
-- (work, served name), the latest curation winning by created DESC, curation_id DESC, as for every field; a value
-- that isn't a JSON array of strings is ignored (the seat keeps the sources' strings). CreateWorksBase serves the list
-- in place of the sources' strings for that seat. A plain view: a handful of rows.
CREATE OR REPLACE VIEW openalex.works.work_curated_affiliation_strings
COMMENT 'The latest raw_affiliation_strings["<served name>"] curation per (work, seat name) (SyncWorkFieldCurations, oxjob #1569). CreateWorksBase serves it as that seat''s raw affiliation strings.'
AS
SELECT work_id, raw_author_name, raw_affiliation_strings, curation_id
FROM (
  SELECT
    work_id,
    regexp_extract(property, '^raw_affiliation_strings\\["(.+)"\\]$', 1) AS raw_author_name,
    from_json(value, 'ARRAY<STRING>')                                   AS raw_affiliation_strings,
    curation_id
  FROM openalex.works.work_field_curations
  WHERE left(property, 25) = 'raw_affiliation_strings["'
    AND action = 'replace'
  QUALIFY ROW_NUMBER() OVER (PARTITION BY work_id, property ORDER BY created DESC, curation_id DESC) = 1
)
WHERE raw_author_name <> '' AND raw_affiliation_strings IS NOT NULL